Thống kê dữ liệu -> làm sạch dữ liệu

In [4]:
import pyarrow.parquet as pq
import os
import glob

def profile_big_data(file_pattern, group_name):
    files = glob.glob(file_pattern)
    total_rows = 0
    total_nulls = 0
    cols = 0
    
    print(f"--- Thống kê nhóm: {group_name} ---")
    
    for f in files:
        # Chỉ đọc metadata của file, không nạp dữ liệu vào RAM
        parquet_file = pq.ParquetFile(f)
        
        # Lấy số hàng từ metadata
        rows = parquet_file.metadata.num_rows
        total_rows += rows
        
        # Lấy số cột
        if cols == 0:
            cols = parquet_file.metadata.num_columns
        
        # Vì không nạp dữ liệu nên không đếm trực tiếp Null từng ô được
        # Nhưng bạn có thể lấy thống kê null từ metadata của từng cột (nếu có)
        for i in range(parquet_file.metadata.num_row_groups):
            row_group = parquet_file.metadata.row_group(i)
            for j in range(row_group.num_columns):
                column_stats = row_group.column(j).statistics
                if column_stats and column_stats.has_null_count:
                    total_nulls += column_stats.null_count
        
    print(f"- Tổng số hàng: {total_rows:,}")
    print(f"- Số lượng cột: {cols}")
    print(f"- Tổng giá trị thiếu (Null): {total_nulls:,}")
    print("-" * 30)

base_path = r"D:\Kì II 2025-2026\BigData\BTL_BigData\Data"

# Chạy thống kê
profile_big_data(os.path.join(base_path, "raw_meta_*.parquet"), "METADATA")
profile_big_data(os.path.join(base_path, "raw_review_*.parquet"), "REVIEW")

--- Thống kê nhóm: METADATA ---
- Tổng số hàng: 12,617,185
- Số lượng cột: 3
- Tổng giá trị thiếu (Null): 1,378,324
------------------------------
--- Thống kê nhóm: REVIEW ---
- Tổng số hàng: 159,879,405
- Số lượng cột: 7
- Tổng giá trị thiếu (Null): 0
------------------------------


Thống kê các giá trị bị null/ thiếu

In [6]:
import pyarrow.parquet as pq
import os
import glob

def profile_detail_per_file(base_path):
    # Lấy tất cả các file parquet trong thư mục
    files = glob.glob(os.path.join(base_path, "*.parquet"))
    
    print(f"{'Tên File':<45} | {'Số hàng':>12} | {'Số cột':>8} | {'Giá trị Null':>12}")
    print("-" * 85)
    
    for f in files:
        file_name = os.path.basename(f)
        try:
            # Đọc metadata của file
            p_file = pq.ParquetFile(f)
            meta = p_file.metadata
            
            num_rows = meta.num_rows
            num_cols = meta.num_columns
            
            # Tính tổng null từ statistics trong metadata
            total_nulls = 0
            for i in range(meta.num_row_groups):
                row_group = meta.row_group(i)
                for j in range(row_group.num_columns):
                    stats = row_group.column(j).statistics
                    if stats and stats.has_null_count:
                        total_nulls += stats.null_count
            
            print(f"{file_name:<45} | {num_rows:>12,} | {num_cols:>8} | {total_nulls:>12,}")
            
        except Exception as e:
            print(f"{file_name:<45} | LỖI: {str(e)}")

# Đường dẫn đến thư mục dữ liệu của bạn
path = r"D:\Kì II 2025-2026\BigData\Data_cleaned"
profile_detail_per_file(path)

Tên File                                      |      Số hàng |   Số cột | Giá trị Null
-------------------------------------------------------------------------------------
clean_meta_All_Beauty.parquet                 |      112,590 |        3 |            0
clean_meta_Amazon_Fashion.parquet             |      826,108 |        3 |            0
clean_meta_Baby_Products.parquet              |      217,724 |        3 |            0
clean_meta_Beauty_and_Personal_Care.parquet   |    1,028,914 |        3 |            0
clean_meta_Clothing_Shoes_and_Jewelry.parquet |    7,218,481 |        3 |            0
clean_meta_Grocery_and_Gourmet_Food.parquet   |      603,274 |        3 |            0
clean_meta_Handmade_Products.parquet          |      164,817 |        3 |            0
clean_meta_Health_and_Household.parquet       |      797,563 |        3 |            0
clean_meta_Health_and_Personal_Care.parquet   |       60,293 |        3 |            0
clean_meta_Sports_and_Outdoors.parquet      

In [7]:
import pyarrow.parquet as pq
import os
import glob

def print_schema_info(file_path, group_name):
    print(f"\n=== SCHEMA CHUẨN: {group_name} ===")
    schema = pq.read_schema(file_path)
    print(f"{'Tên cột':<20} | {'Kiểu dữ liệu':<20}")
    print("-" * 45)
    for field in schema:
        print(f"{field.name:<20} | {str(field.type):<20}")

base_path = r"D:\Kì II 2025-2026\BigData\BTL_BigData\Data"

# Lấy 1 file đại diện cho mỗi nhóm để xem Schema
meta_sample = glob.glob(os.path.join(base_path, "raw_meta_*.parquet"))[0]
review_sample = glob.glob(os.path.join(base_path, "raw_review_*.parquet"))[0]

print_schema_info(meta_sample, "METADATA")
print_schema_info(review_sample, "REVIEW")


=== SCHEMA CHUẨN: METADATA ===
Tên cột              | Kiểu dữ liệu        
---------------------------------------------
parent_asin          | large_string        
main_category        | large_string        
title                | large_string        

=== SCHEMA CHUẨN: REVIEW ===
Tên cột              | Kiểu dữ liệu        
---------------------------------------------
parent_asin          | large_string        
user_id              | large_string        
rating               | double              
text                 | large_string        
timestamp            | int64               
verified_purchase    | bool                
helpful_vote         | int64               


In [8]:
def check_all_files_consistency(file_pattern, group_label):
    files = glob.glob(file_pattern)
    if not files: return
    
    # Lấy file đầu tiên làm mốc so sánh
    ref_schema = pq.read_schema(files[0])
    ref_dict = {field.name: field.type for field in ref_schema}
    
    inconsistent_count = 0
    print(f"\n--- KIỂM TRA ĐỘ ĐỒNG NHẤT NHÓM: {group_label} ---")
    
    for f in files:
        curr_schema = pq.read_schema(f)
        curr_dict = {field.name: field.type for field in curr_schema}
        
        errors = []
        for col, dtype in ref_dict.items():
            if col not in curr_dict:
                errors.append(f"Thiếu cột [{col}]")
            elif curr_dict[col] != dtype:
                errors.append(f"Cột [{col}] lệch kiểu: {curr_dict[col]} (Chuẩn: {dtype})")
        
        if errors:
            inconsistent_count += 1
            print(f"File {os.path.basename(f)}: {', '.join(errors)}")
            
    if inconsistent_count == 0:
        print(f"=> Kết quả: 100% file trong nhóm {group_label} đồng nhất về kiểu dữ liệu.")
    else:
        print(f"=> Cảnh báo: Có {inconsistent_count} file bị lệch Schema.")

# Chạy kiểm tra cho cả 2 loại file
check_all_files_consistency(os.path.join(base_path, "raw_meta_*.parquet"), "METADATA")
check_all_files_consistency(os.path.join(base_path, "raw_review_*.parquet"), "REVIEW")


--- KIỂM TRA ĐỘ ĐỒNG NHẤT NHÓM: METADATA ---
=> Kết quả: 100% file trong nhóm METADATA đồng nhất về kiểu dữ liệu.

--- KIỂM TRA ĐỘ ĐỒNG NHẤT NHÓM: REVIEW ---
=> Kết quả: 100% file trong nhóm REVIEW đồng nhất về kiểu dữ liệu.


TIỀN XỬ LÝ DỮ LIỆU

In [ ]:
# CÁC KỸ THUẬT CẦN DÙNG

# Xử lý Missing Values

# Text Cleaning 

# chuyển về chữ thường

# loại bỏ ký tự đặc biệt

# loại bỏ stop works

# chuẩn hóa từ


In [ ]:
pip install nltk

In [5]:
import polars as pl
import os
import re
import nltk

# 1. CHUẨN BỊ SIÊU REGEX STOPWORDS (TỐI ƯU TỐC ĐỘ)

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# Lấy danh sách stopwords mặc định
raw_stops = stopwords.words('english')

# Làm sạch stopwords: bỏ dấu nháy đơn (vd: don't -> dont) để khớp với text sau khi clean
clean_stops = set([re.sub(r'[^a-z0-9]', '', w) for w in raw_stops])
clean_stops.discard('') # Xóa phần tử rỗng nếu có

# Ép thành chuỗi Regex khổng lồ có dạng: \b(i|me|my|dont|is|are...)\b
# (?i) là không phân biệt hoa thường, \b là giới hạn ranh giới từ
STOP_WORDS_PATTERN = r"(?i)\b(" + "|".join(clean_stops) + r")\b"

# ==========================================
# 2. CẤU HÌNH THƯ MỤC & DANH MỤC
# ==========================================
RAW_DIR = r"D:\Kì II 2025-2026\BTL_BIGDATA\Data"
CLEAN_DIR = r"D:\Kì II 2025-2026\BigData\\BTL_BIGDATA\Data_cleaned"

CATEGORIES = [
    # "All_Beauty",
    # "Amazon_Fashion",
    # "Handmade_Products",
    # "Health_and_Personal_Care",
    # "Baby_Products",
    "Grocery_and_Gourmet_Food",
    "Sports_and_Outdoors",
    "Beauty_and_Personal_Care",
    "Health_and_Household",
    "Clothing_Shoes_and_Jewelry"
]

os.makedirs(CLEAN_DIR, exist_ok=True)

# ==========================================
# 3. HÀM XỬ LÝ CHÍNH (100% POLARS NATIVE API)
# ==========================================
def preprocess_superfast(category):
    print(f"\n{'-'*60}")
    print(f"ĐANG LÀM SẠCH: {category}")
    print(f"{'-'*60}")

    raw_meta = os.path.join(RAW_DIR, f"raw_meta_{category}.parquet")
    raw_review = os.path.join(RAW_DIR, f"raw_review_{category}.parquet")
    
    clean_meta = os.path.join(CLEAN_DIR, f"clean_meta_{category}.parquet")
    clean_review = os.path.join(CLEAN_DIR, f"clean_review_{category}.parquet")

    # ------------------------------------------
    # A. XỬ LÝ METADATA
    # ------------------------------------------
    if os.path.exists(raw_meta):
        if not os.path.exists(clean_meta):
            print(f"⏳ Đang xử lý Meta data...")
            (
                pl.scan_parquet(raw_meta)
                .drop_nulls(subset=["parent_asin"])
                .with_columns([
                    pl.col("main_category").fill_null("Unknown_Category"),
                    pl.col("title").fill_null("Unknown_Title")
                ])
                .sink_parquet(clean_meta)
            )
            print(f"Đã lưu Meta sạch: {clean_meta}")
        else:
            print(f"Bỏ qua Meta: Đã tồn tại.")
    else:
        print(f"Không tìm thấy file gốc: {raw_meta}")

    # ------------------------------------------
    # B. XỬ LÝ REVIEW (ĐƯỜNG ỐNG NLP THUẦN POLARS)
    # ------------------------------------------
    if os.path.exists(raw_review):
        if not os.path.exists(clean_review):
            print(f"Đang dọn dẹp Text & Stopwords...")
            (
                pl.scan_parquet(raw_review)
                # 1. XỬ LÝ MISSING VALUES
                .drop_nulls(subset=["parent_asin", "user_id"])
                .with_columns([
                    pl.col("text").fill_null(""),
                    pl.col("helpful_vote").fill_null(0),
                    pl.col("verified_purchase").fill_null(False)
                ])
                
                # 2. XỬ LÝ TEXT LIÊN HOÀN (VECTORIZED RE-ENGINEERING)
                .with_columns([
                    pl.col("text")
                    .str.to_lowercase()                           # 1. Hạ chữ thường
                    .str.replace_all(r"<[^>]+>", " ")             # 2. Xóa sạch thẻ HTML (<br>, <b>)
                    .str.replace_all(r"'", "")                    # 3. Bỏ dấu nháy đơn (don't -> dont)
                    .str.replace_all(r"[^a-z0-9\s]", " ")         # 4. Ký tự đặc biệt -> khoảng trắng
                    .str.replace_all(STOP_WORDS_PATTERN, "")      # 5. Xóa Stopwords bằng Siêu Regex
                    .str.replace_all(r"\s+", " ")                 # 6. Gom khoảng trắng thừa
                    .str.strip_chars()                            # 7. Cắt khoảng trắng 2 đầu
                    .alias("text_cleaned")                        # Lưu kết quả sang cột mới
                ])
                .sink_parquet(clean_review)
            )
            print(f"Đã lưu Review sạch: {clean_review}")
        else:
            print(f"⏭Bỏ qua Review: Đã tồn tại.")
    else:
        print(f"Không tìm thấy file gốc: {raw_review}")

# ==========================================
# 4. THỰC THI CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    for cat in CATEGORIES:
        preprocess_superfast(cat)
        
    print("\n HOÀN TẤT !")
    print(f"Kiểm tra kết quả tại: {CLEAN_DIR}")


------------------------------------------------------------
🚀 ĐANG LÀM SẠCH SIÊU TỐC: Grocery_and_Gourmet_Food
------------------------------------------------------------
⏳ Đang xử lý Meta data...
Đã lưu Meta sạch: D:\Kì II 2025-2026\BigData\\BTL_BIGDATA\Data_cleaned\clean_meta_Grocery_and_Gourmet_Food.parquet
Đang dọn dẹp Text & Stopwords...
Đã lưu Review sạch: D:\Kì II 2025-2026\BigData\\BTL_BIGDATA\Data_cleaned\clean_review_Grocery_and_Gourmet_Food.parquet

------------------------------------------------------------
🚀 ĐANG LÀM SẠCH SIÊU TỐC: Sports_and_Outdoors
------------------------------------------------------------
⏳ Đang xử lý Meta data...
Đã lưu Meta sạch: D:\Kì II 2025-2026\BigData\\BTL_BIGDATA\Data_cleaned\clean_meta_Sports_and_Outdoors.parquet
Đang dọn dẹp Text & Stopwords...
Đã lưu Review sạch: D:\Kì II 2025-2026\BigData\\BTL_BIGDATA\Data_cleaned\clean_review_Sports_and_Outdoors.parquet

------------------------------------------------------------
🚀 ĐANG LÀM SẠCH SIÊ